# Task B Dataset Overview

Task B is different from A and C because:

Task A → labels from WorldCover (automatic)

Task C → labels from OSM (semi-automatic)

Task B → labels from YOU (manual annotation)

You must personally draw dots
on rooftop centers in Label Studio
No automation possible
This is called manual annotation

Task B Plan

Step 1 → Export city tiles from GEE,
          High resolution 10m tiles,
          5 Pakistani cities

Step 2 → Set up Label Studio,
          Free annotation tool,
          Runs in Colab browser

Step 3 → Annotate rooftop centroids,
          Draw one dot per rooftop,
          Target: 500 tiles

Step 4 → Convert annotations,
          Dots → Gaussian density maps,
          .json → .npy files


Sum of density map = number of houses

This is the key property of CSRNet:
Model predicts density map
You sum the entire map
That sum = house count

Example:
Tile has 50 houses
Density map sum = 50.0
Model predicted sum = 48.3
Error = 50 - 48.3 = 1.7 houses
This is your MAE metric

Task B Dataset Overview

What you need:

─────────────────────────────────────

Input tiles:
→ High resolution city images
→ 10 metres per pixel
→ 512×512 pixels
→ 5 Pakistani cities

Output density maps:
→ Same size 512×512
→ Float values
→ Sum = house count
→ Saved as .npy files

Ground truth:
→ You manually mark rooftop dots
→ One dot per house centroid
→ In Label Studio software
→ Target: 100 tiles per city
→ 5 cities × 100 = 500 tiles

Revised Task B Plan

OLD PLAN:

500 tiles × 5 cities = too much work

NEW PLAN:

20 tiles per city
5 cities
100 tiles total
Split annotation among 4 team members
Each person annotates 25 tiles

# Export City Tiles From GEE

Task A — Land cover:
You classify entire regions like
Forest, desert, cropland.
These are LARGE areas,
Hundreds of metres wide,
100m per pixel is enough,
You can see a forest at 100m resolution.

Task B — House counting:
A single house in Pakistan
is roughly 10m × 10m.
At 100m resolution:
one house = one tiny pixel
impossible to see or count

At 10m resolution:
one house = roughly 1×1 pixel
visible and countable.

This is why resolution matters:
Object size must be larger than
your pixel size to be detectable


Simple Rule

Object size ≥ pixel size to be visible

House = 10m × 10m
Need pixel ≤ 10m to see it

Forest = kilometres wide
100m pixel is fine

# Mount and connect:

In [ ]:
from google.colab import drive, auth
import ee

drive.mount('/content/drive')
auth.authenticate_user()
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='geovisionpakistan')

print("Ready")

Mounted at /content/drive
Ready


# Export 5 city tiles

In [ ]:
import time

# 5 Pakistani cities bounding boxes
cities = {
    "lahore"    : [74.20, 31.40, 74.50, 31.65],
    "karachi"   : [66.90, 24.80, 67.20, 25.05],
    "islamabad" : [72.95, 33.60, 73.20, 33.78],
    "peshawar"  : [71.45, 34.00, 71.70, 34.20],
    "faisalabad": [73.00, 31.25, 73.25, 31.50],
}

tasks = {}

for city, bbox in cities.items():

    region = ee.Geometry.Rectangle(bbox)

    img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterBounds(region)
             .filterDate("2022-01-01", "2022-12-31")
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 5))
             .sort("CLOUDY_PIXEL_PERCENTAGE")
             .first()
             .select(["B4", "B3", "B2"]))   # RGB only

    task = ee.batch.Export.image.toDrive(
        image       = img,
        description = f"city_{city}",
        region      = region,
        scale       = 10,          # 10m resolution for house detail
        crs         = "EPSG:32642",
        maxPixels   = 1e9,
        fileFormat  = "GeoTIFF",
        folder      = "GeoVision exports"
    )

    task.start()
    tasks[city] = task.id
    print(f"Started: {city} | ID: {task.id}")

Started: lahore | ID: HX5Q7744TWBEW7N6745SQ4L6
Started: karachi | ID: TINV3UA3GZUFKBTLRLHAX2VR
Started: islamabad | ID: PWAW3XWZNZ3KUN7HWUCVZTHI
Started: peshawar | ID: JWVPO2SG2HHVTYSSN3SPY44H
Started: faisalabad | ID: 6YSOJ2AARDTLQ7CYYTGDNOZR


#  Monitor all cities

In [ ]:
start = time.time()

while True:
    all_done = True
    elapsed  = time.time() - start
    mins, secs = divmod(int(elapsed), 60)

    for city, task_id in tasks.items():
        t     = [x for x in ee.batch.Task.list()
                 if x.id == task_id][0]
        state = t.status()['state']
        print(f"[{mins:02d}:{secs:02d}] {city}: {state}")

        if state not in ('COMPLETED','FAILED','CANCELLED'):
            all_done = False

    if all_done:
        break

    print("---")
    time.sleep(60)

print("All city exports done")

[00:00] lahore: RUNNING
[00:00] karachi: RUNNING
[00:00] islamabad: RUNNING
[00:00] peshawar: READY
[00:00] faisalabad: READY
---
[01:01] lahore: RUNNING
[01:01] karachi: RUNNING
[01:01] islamabad: COMPLETED
[01:01] peshawar: RUNNING
[01:01] faisalabad: READY
---
[02:03] lahore: COMPLETED
[02:03] karachi: COMPLETED
[02:03] islamabad: COMPLETED
[02:03] peshawar: RUNNING
[02:03] faisalabad: RUNNING
---
[03:05] lahore: COMPLETED
[03:05] karachi: COMPLETED
[03:05] islamabad: COMPLETED
[03:05] peshawar: RUNNING
[03:05] faisalabad: RUNNING
---
[04:06] lahore: COMPLETED
[04:06] karachi: COMPLETED
[04:06] islamabad: COMPLETED
[04:06] peshawar: COMPLETED
[04:06] faisalabad: RUNNING
---
[05:08] lahore: COMPLETED
[05:08] karachi: COMPLETED
[05:08] islamabad: COMPLETED
[05:08] peshawar: COMPLETED
[05:08] faisalabad: COMPLETED
All city exports done


# Find And Fix Issues

In [ ]:
import os

print("Searching entire Drive for city files...")

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.startswith("city_") and file.endswith(".tif"):
            full_path = os.path.join(root, file)
            size_mb   = os.path.getsize(full_path) / 1e6
            print(f"{full_path} — {size_mb:.1f} MB")

Searching entire Drive for city files...
/content/drive/MyDrive/GeoVision exports (1)/city_lahore.tif — 36.9 MB
/content/drive/MyDrive/GeoVision exports/city_islamabad.tif — 26.1 MB
/content/drive/MyDrive/GeoVision exports/city_karachi.tif — 0.8 MB
/content/drive/MyDrive/GeoVision exports/city_peshawar.tif — 28.1 MB
/content/drive/MyDrive/GeoVision exports/city_faisalabad.tif — 38.0 MB


Issue 1 — Lahore In Wrong Folder

Lahore saved to:
"GeoVision exports (1)"  ← different folder

Others saved to:
"GeoVision exports"      ← correct folder

GEE created a duplicate folder with (1) suffix. This happens when Drive has naming conflicts.

Issue 2 — Karachi Is Only 0.8 MB

Lahore:     36.9 MB  ✓ normal

Islamabad:  26.1 MB  ✓ normal

Karachi:     0.8 MB  ✗ too small

Peshawar:   28.1 MB  ✓ normal

Faisalabad: 38.0 MB  ✓ normal

Karachi file is suspiciously small. Likely exported with mostly empty/ocean pixels or cloud cover.



Issue 1 — Move Lahore to correct folder:

In [ ]:
import shutil, os

src = "/content/drive/MyDrive/GeoVision exports (1)/city_lahore.tif"
dst = "/content/drive/MyDrive/GeoVision exports/city_lahore.tif"

shutil.copy(src, dst)
print("Lahore moved to correct folder")

Lahore moved to correct folder


Issue 2 — Inspect Karachi file:

In [ ]:
import rasterio
import numpy as np

karachi_path = "/content/drive/MyDrive/GeoVision exports/city_karachi.tif"

with rasterio.open(karachi_path) as src:
    print(f"Shape : {src.count} bands, {src.height} rows, {src.width} cols")
    print(f"CRS   : {src.crs}")
    print(f"Scale : {src.res}")

    data = src.read()
    print(f"Min   : {data.min()}")
    print(f"Max   : {data.max()}")
    print(f"Zeros : {np.sum(data==0)/data.size:.1%}")

Shape : 3 bands, 2813 rows, 3071 cols
CRS   : EPSG:32642
Scale : (10.0, 10.0)
Min   : 0
Max   : 4920
Zeros : 98.7%


Karachi zeros: 98.7%

Almost entire image is empty.
Only 1.3% has real data.
Completely unusable for annotation.

Why This Happened

Karachi bounding box we used:
[66.90, 24.80, 67.20, 25.05]

Karachi sits on the coast.
Large portion of this bounding box
is Arabian Sea not land.

Sea pixels = 0 (no satellite data)
98.7% of box = sea
Only 1.3% = actual Karachi land

Fix — Re-export Karachi With Better Bbox

In [ ]:
import ee, time

# Tighter bbox — inland Karachi only
# Avoids Arabian Sea
region = ee.Geometry.Rectangle(
    [67.00, 24.85, 67.25, 25.05]
)

img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(region)
         .filterDate("2022-01-01", "2022-12-31")
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 10))
         .sort("CLOUDY_PIXEL_PERCENTAGE")
         .first()
         .select(["B4", "B3", "B2"]))

task = ee.batch.Export.image.toDrive(
    image       = img,
    description = "city_karachi_fixed",
    region      = region,
    scale       = 10,
    crs         = "EPSG:32642",
    maxPixels   = 1e9,
    fileFormat  = "GeoTIFF",
    folder      = "GeoVision exports"
)

task.start()
print(f"Karachi re-export started: {task.id}")

# Monitor
start = time.time()
while True:
    t      = [x for x in ee.batch.Task.list()
               if x.id == task.id][0]
    state  = t.status()['state']
    elapsed = time.time() - start
    mins, secs = divmod(int(elapsed), 60)
    print(f"[{mins:02d}:{secs:02d}] {state}")
    if state in ('COMPLETED','FAILED','CANCELLED'):
        break
    time.sleep(30)

print("Done")

Karachi re-export started: RGMR7QFFJJB34IUUUUCWRCR7
[00:00] READY
[00:30] RUNNING
[01:00] RUNNING
[01:31] RUNNING
[02:01] COMPLETED
Done


Now Verify All 5 Cities Together

In [ ]:
import rasterio
import numpy as np
import os

base = "/content/drive/MyDrive/GeoVision exports"

cities = {
    "lahore"    : f"{base}/city_lahore.tif",
    "islamabad" : f"{base}/city_islamabad.tif",
    "peshawar"  : f"{base}/city_peshawar.tif",
    "faisalabad": f"{base}/city_faisalabad.tif",
    "karachi"   : f"{base}/city_karachi_fixed.tif",
}

print("City File Verification")
print("="*55)

all_good = True

for city, path in cities.items():

    if not os.path.exists(path):
        print(f"{city:12s} → FILE NOT FOUND ✗")
        all_good = False
        continue

    with rasterio.open(path) as src:
        data     = src.read()
        zeros    = np.sum(data == 0) / data.size
        size_mb  = os.path.getsize(path) / 1e6
        status   = "✓" if zeros < 0.30 else "✗ too many zeros"

        print(f"{city:12s} → {src.height}×{src.width} | zeros: {zeros:.1%} | {size_mb:.1f}MB | {status}")

        if zeros >= 0.30:
            all_good = False

print("="*55)
if all_good:
    print("All 5 cities ready for annotation ✓")
else:
    print("Some cities need fixing ✗")

City File Verification
lahore       → 2917×2990 | zeros: 8.5% | 36.9MB | ✓
islamabad    → 2090×2399 | zeros: 0.0% | 26.1MB | ✓
peshawar     → 2277×2364 | zeros: 0.0% | 28.1MB | ✓
faisalabad   → 2864×2486 | zeros: 0.0% | 38.0MB | ✓
karachi      → 2251×2557 | zeros: 96.3% | 1.3MB | ✗ too many zeros
Some cities need fixing ✗


Fix The Karachi Bbox Properly

The issue is Karachi's urban area is very specific.

In [ ]:
import ee, time

# Very specific inland Karachi
# Gulshan, PECHS, Saddar area — dense urban
region = ee.Geometry.Rectangle(
    [67.02, 24.87, 67.15, 24.97]
)

# Preview what this covers first
print("Bbox area:")
print("West : 67.02°E")
print("East : 67.15°E")
print("South: 24.87°N")
print("North: 24.97°N")

img = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
         .filterBounds(region)
         .filterDate("2022-01-01", "2022-12-31")
         .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
         .sort("CLOUDY_PIXEL_PERCENTAGE")
         .first()
         .select(["B4", "B3", "B2"]))

task = ee.batch.Export.image.toDrive(
    image       = img,
    description = "city_karachi_v3",
    region      = region,
    scale       = 10,
    crs         = "EPSG:32642",
    maxPixels   = 1e9,
    fileFormat  = "GeoTIFF",
    folder      = "GeoVision exports"
)

task.start()
print(f"\nKarachi v3 started: {task.id}")

start = time.time()
while True:
    t     = [x for x in ee.batch.Task.list()
              if x.id == task.id][0]
    state = t.status()['state']
    elapsed = time.time() - start
    mins, secs = divmod(int(elapsed), 60)
    print(f"[{mins:02d}:{secs:02d}] {state}")
    if state in ('COMPLETED','FAILED','CANCELLED'):
        break
    time.sleep(30)

print("Done")

Bbox area:
West : 67.02°E
East : 67.15°E
South: 24.87°N
North: 24.97°N

Karachi v3 started: VYBZOPGKVDHRPCKDIMMVM2IN
[00:00] READY
[00:30] RUNNING
[01:00] COMPLETED
Done


Verify Karachi v3:

In [ ]:
import rasterio
import numpy as np

path = "/content/drive/MyDrive/GeoVision exports/city_karachi_v3.tif"

with rasterio.open(path) as src:
    data  = src.read()
    zeros = np.sum(data == 0) / data.size
    size  = os.path.getsize(path) / 1e6
    print(f"Shape : {src.height}×{src.width}")
    print(f"Zeros : {zeros:.1%}")
    print(f"Size  : {size:.1f} MB")
    print(f"Status: {'✓ Good' if zeros < 0.30 else '✗ Too many zeros'}")

Shape : 1127×1330
Zeros : 72.0%
Size  : 2.3 MB
Status: ✗ Too many zeros


Decision — Skip Karachi

We tried 3 times:
Attempt 1 → 98.7% zeros ✗
Attempt 2 → 96.3% zeros ✗
Attempt 3 → 72.0% zeros ✗

Karachi sits on Arabian Sea coast.
Finding a bbox with enough
inland urban area is difficult
without a detailed map.

Decision: Skip Karachi
Use 4 cities instead of 5

4 cities × 25 tiles = 100 tiles
Target still met
No time wasted further

Your 4 Cities Are Ready

Lahore      2917×2990

Islamabad   2090×2399

Peshawar    2277×2364

Faisalabad  2864×2486

You have 5 files on Drive:
city_lahore.tif      ← use this

city_islamabad.tif   ← use this

city_peshawar.tif    ← use this

city_faisalabad.tif  ← use this

city_karachi.tif     ← ignore this

city_karachi_fixed   ← ignore this

city_karachi_v3      ← ignore this

When you write tiling code
just do not include Karachi path
in your cities dictionary
That is all

You mark rooftop dots in Label Studio.
Each dot = one house center.

To convert dots into density map
you place a smooth circular blob
around each dot.

All blobs added together = density map.

This blob is called a Gaussian distribution = bell shaped curve

In density maps:
Each rooftop dot → place Gaussian blob

Blob is bright in center (where house is)

Blob fades smoothly outward

All blobs added = density map

Sum of map = number of houses

# Cut City Tiles

Before Label Studio you need tiles to annotate.

In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
from PIL import Image
import os, time

base    = "/content/drive/MyDrive/GeoVision exports"
out_dir = f"{base}/task_b/tiles_for_annotation"
os.makedirs(out_dir, exist_ok=True)

cities = {
    "lahore"    : f"{base}/city_lahore.tif",
    "islamabad" : f"{base}/city_islamabad.tif",
    "peshawar"  : f"{base}/city_peshawar.tif",
    "faisalabad": f"{base}/city_faisalabad.tif",
}

TILE_SIZE = 256
STRIDE    = 256    # no overlap for annotation tiles
MAX_TILES = 25     # 25 tiles per city

total_saved = 0
start       = time.time()

for city, path in cities.items():

    city_saved = 0

    with rasterio.open(path) as src:
        H = src.height
        W = src.width

        for row in range(0, H - TILE_SIZE, STRIDE):
            for col in range(0, W - TILE_SIZE, STRIDE):

                if city_saved >= MAX_TILES:
                    break

                window = Window(col, row, TILE_SIZE, TILE_SIZE)
                patch  = src.read(window=window)

                # Skip empty tiles
                if np.sum(patch == 0) / patch.size > 0.30:
                    continue

                # Normalize and save
                rgb = np.clip(patch / 3000.0 * 255, 0, 255)
                rgb = rgb.astype(np.uint8)

                img = Image.fromarray(rgb.transpose(1, 2, 0))
                img.save(f"{out_dir}/{city}_{city_saved:03d}.jpg")

                city_saved  += 1
                total_saved += 1

            if city_saved >= MAX_TILES:
                break

    elapsed    = time.time() - start
    mins, secs = divmod(int(elapsed), 60)
    print(f"[{mins:02d}:{secs:02d}] {city}: {city_saved} tiles saved")

print(f"\nTotal tiles for annotation: {total_saved}")

[00:00] lahore: 25 tiles saved
[00:01] islamabad: 25 tiles saved
[00:01] peshawar: 25 tiles saved
[00:02] faisalabad: 25 tiles saved

Total tiles for annotation: 100


This tiling code uses STRIDE=256
which equals TILE_SIZE=256.

This means NO overlap between tiles.

Task A used STRIDE=112 (50% overlap).

Why does annotation tiling
use NO overlap?

If STRIDE = 112 (overlap):
Same rooftop appears in 2 tiles
You annotate it in Tile 1 ✓
You annotate it in Tile 2 ✓

Now you have same house counted TWICE
Density map sum = 2
But real count = 1

Model learns wrong count
Training data is corrupted

If STRIDE = 256 (no overlap):
Each rooftop appears in exactly 1 tile
You annotate it once ✓
Density map sum = 1
Real count = 1 ✓
Perfect match

Simple rule:

Training tiles → overlap OK
                 more data = better learning

Annotation tiles → NO overlap
                   each object counted once only

# New Section